# 02 — Tuning binary trên client Dirichlet

DNN 10 + CNN 10 = 20 trial. Mỗi trial chạy 3 Local + Centralized + FedAvg theo số mẫu fit thực tế của client mới. Chọn theo F1 Attack trung bình 12 đánh giá validation: 6 Local chéo, 3 Centralized và 3 FL; tie-break F1 FL client thấp nhất rồi recall trung bình. Không dùng test.

Giữ anchor/search space như bản sao trước và project gốc: DNN 8-4-2/dropout 0,5/batch 1024/LR 0,0005; CNN 8-16-8/kernel 3/dropout 0,2/batch 2048/LR 0,0005. FL 15 round × 3 local epoch; Local/Centralized tối đa 45 epoch, patience 5. Không dùng tham số hoặc trọng số đã chọn ở thí nghiệm cũ.

In [1]:
from pathlib import Path
import os, sys, json, importlib
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p/'COPY_PROVENANCE.json').is_file()
               and (p/'configs/tuning.json').is_file() and (p/'notebook_workflow.py').is_file())
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
for module_name in ('common', 'feature_preprocessing', 'prepare', 'models', 'metrics', 'notebook_workflow'):
    module = importlib.import_module(module_name)
    assert Path(module.__file__).resolve().parent == PROJECT.resolve(), 'Restart kernel to load this copy.'
    importlib.reload(module)
workflow = sys.modules['notebook_workflow']
CONFIG = PROJECT/'configs/tuning.json'
C = workflow.settings(CONFIG)
RUN_ROOT = workflow.run_root(C)
print('Project:', PROJECT)
print('Results:', RUN_ROOT)
display(workflow.budget(CONFIG))

Project: .
Results: runs/cic_all_attack_dirichlet_a05_max1p5m


{'tasks': ['binary'],
 'tuning_task': 'binary',
 'models': ['dnn', 'cnn'],
 'trials_per_model': 10,
 'rounds': 15,
 'local_epochs': 3,
 'baseline_epochs': 45,
 'client_optimizer': 'adam',
 'server_optimizer': 'sgd',
 'server_learning_rate': 1.0,
 'tuning_weighting': 'number_of_examples',
 'final_weightings': ['number_of_examples'],
 'selection': 'mean_cross_client_validation_f1',
 'checkpoint_selection': 'final_round',
 'total_tuning_trials': 20,
 'fits_per_trial': '3 Local + 1 Centralized + 1 sample-weighted FedAvg',
 'objective': 'mean of 12 validation F1: 6 cross-client Local + 3 Centralized + 3 FL',
 'trial_1': {'dnn': {'architecture': '8-4-2',
   'dropout': 0.5,
   'batch_size': 1024,
   'client_learning_rate': 0.0005},
  'cnn': {'architecture': '8-16-8',
   'kernel_size': 3,
   'dropout': 0.2,
   'batch_size': 2048,
   'client_learning_rate': 0.0005}}}

In [2]:
MAX_NEW_EVALUATIONS = None  # Đặt 1 để chạy từng trial.

In [3]:
for model in C["models"]:
    display(workflow.tune(model,CONFIG,max_new_evaluations=MAX_NEW_EVALUATIONS,task="binary"))

[I 2026-11-03 16:21:15,920] A new study created in RDB with name: binary_dnn
TUNE binary/dnn trial 0: {'architecture': '8-4-2', 'dropout': 0.5, 'batch_size': 1024, 'client_learning_rate': 0.0005}
2026-11-03 16:21:16.925578: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-11-03 16:21:16.925669: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-11-03 16:21:16.926800: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-11-03 16:21:21.238082: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plug

{'study_identity': 'c8bc1131f6b0f873a3ee05ea3923ee3aaf3239d654f0a23d8fe455faac5633f8',
 'task': 'binary',
 'model': 'dnn',
 'trial': 14,
 'params': {'architecture': '16-8-4',
  'dropout': 0.3,
  'batch_size': 4096,
  'client_learning_rate': 0.0005},
 'rank': [0.7622448240725608, 0.6908694297288875, 0.7734320253593564],
 'completed_trials': 10,
 'required_trials': 10,
 'ready_for_final': True,
 'test_used_for_selection': False,
 'tuning_scenario': 'fedavg_weighted',
 'tuning_scenarios': ['local_cic_1',
  'local_cic_2',
  'local_cic_3',
  'centralized',
  'fedavg_weighted'],
 'objective': 'mean_cross_client_validation_f1'}

[I 2026-11-03 17:41:24,884] A new study created in RDB with name: binary_cnn
TUNE binary/cnn trial 0: {'architecture': '8-16-8', 'dropout': 0.2, 'batch_size': 2048, 'client_learning_rate': 0.0005, 'kernel_size': 3}
2026-11-03 17:41:25.543349: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-11-03 17:41:25.543400: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-11-03 17:41:25.543427: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-11-03 17:41:27.570763: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to regist

{'study_identity': 'c8bc1131f6b0f873a3ee05ea3923ee3aaf3239d654f0a23d8fe455faac5633f8',
 'task': 'binary',
 'model': 'cnn',
 'trial': 6,
 'params': {'architecture': '4-8-4',
  'dropout': 0.3,
  'batch_size': 1024,
  'client_learning_rate': 0.0003,
  'kernel_size': 5},
 'rank': [0.7854105054080948, 0.6883147295293107, 0.8048760419764819],
 'completed_trials': 10,
 'required_trials': 10,
 'ready_for_final': True,
 'test_used_for_selection': False,
 'tuning_scenario': 'fedavg_weighted',
 'tuning_scenarios': ['local_cic_1',
  'local_cic_2',
  'local_cic_3',
  'centralized',
  'fedavg_weighted'],
 'objective': 'mean_cross_client_validation_f1'}

In [4]:
for model in C["models"]:
    print(model)
    display(workflow.tuning_table(model,CONFIG,task="binary"))

dnn


,trial,score,min_fl_f1,architecture,dropout,batch_size,client_learning_rate
0,0,0.663049,0.345886,8-4-2,0.5,1024,0.0005
1,1,0.727420,0.684808,12-6-3,0.3,2048,0.0020
2,2,0.611919,0.351508,8-4-2,0.5,4096,0.0020
3,3,0.673346,0.401733,12-6-3,0.4,1024,0.0010
4,4,0.707804,0.529003,16-8-4,0.5,2048,0.0005
5,7,0.721521,0.603647,16-8-4,0.3,4096,0.0010
6,8,0.691108,0.369748,12-6-3,0.4,2048,0.0020
7,10,0.686896,0.396252,8-4-2,0.3,1024,0.0010
8,14,0.762245,0.690869,16-8-4,0.3,4096,0.0005
9,15,0.700941,0.438841,12-6-3,0.4,2048,0.0005


cnn


,trial,score,min_fl_f1,architecture,dropout,batch_size,client_learning_rate,kernel_size
0,0,0.777620,0.692284,8-16-8,0.2,2048,0.0005,3
1,1,0.777780,0.693379,8-16-8,0.1,2048,0.0010,3
2,2,0.780775,0.702405,16-32-16,0.1,1024,0.0010,5
3,3,0.771475,0.698445,16-32-16,0.3,1024,0.0005,5
4,4,0.773874,0.693321,8-16-8,0.1,4096,0.0005,5
5,6,0.785411,0.688315,4-8-4,0.3,1024,0.0003,5
6,7,0.774221,0.688716,4-8-4,0.3,4096,0.0003,5
7,8,0.733027,0.690550,4-8-4,0.3,1024,0.0003,3
8,9,0.780864,0.689763,4-8-4,0.2,1024,0.0003,5
9,10,0.741447,0.689848,4-8-4,0.3,2048,0.0003,3
